# Análisis exploratorio Music Brainz

In [2]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("eda_musicbrainz")
    .config("spark.driver.memory", "2g")
    .enableHiveSupport()
    .getOrCreate()
)

RAW = "/Obligatorio/landing/musicbrainz"
def I(): return T.IntegerType()
def S(): return T.StringType()
def fld(n, t): return T.StructField(n, t, True)

schemas = {
 "area":          [fld("id",I()),fld("mbid",S()),fld("name",S())],
 "artist":        [fld("id",I()),fld("mbid",S()),fld("name",S()),fld("sort_name",S()),
                   fld("begin_year",I()),fld("end_year",I()),fld("type",I()),fld("area",I())],
 "artist_type":   [fld("id",I()),fld("name",S())],
 "country_area":  [fld("area_id",I())],
 "event":         [fld("id",I()),fld("mbid",S()),fld("name",S()),fld("begin_year",I()),
                   fld("begin_month",I()),fld("begin_day",I()),fld("end_year",I()),
                   fld("end_month",I()),fld("end_day",I()),fld("type",I()),fld("cancelled",S())],
 "event_alias":   [fld("id",I()),fld("event_id",I()),fld("name",S()),fld("locale",S()),
                   fld("type",S()),fld("sort_name",S())],
 "event_type":    [fld("id",I()),fld("name",S())],
 "l_area_area":   [fld("id",I()),fld("parent_area_id",I()),fld("child_area_id",I())],
 "l_artist_event":[fld("id",I()),fld("artist_id",I()),fld("event_id",I())],
 "l_event_place": [fld("id",I()),fld("event_id",I()),fld("place_id",I())],
 "l_event_series":[fld("id",I()),fld("event_id",I()),fld("series_id",I())],
 "place":         [fld("id",I()),fld("mbid",S()),fld("name",S()),fld("type",I()),
                   fld("area",I()),fld("coordinates",S())],
 "series":        [fld("id",I()),fld("mbid",S()),fld("name",S()),fld("type",I())],
}

mb = {}
for name, fields in schemas.items():
    mb[name] = (spark.read.option("header", True)
                .schema(T.StructType(fields))
                .csv(f"{RAW}/mb_{name}.csv"))
print("Tablas cargadas:", list(mb.keys()))



Tablas cargadas: ['area', 'artist', 'artist_type', 'country_area', 'event', 'event_alias', 'event_type', 'l_area_area', 'l_artist_event', 'l_event_place', 'l_event_series', 'place', 'series']


In [3]:
for name, df in mb.items():
    print(f"{name:16} filas={df.count():>8}  columnas={len(df.columns)}")
# Vista previa de las tablas nuevas
for name in ["place", "l_event_place", "l_area_area", "series", "l_event_series"]:
    print(f"\n--- {name} ---")
    mb[name].show(4, truncate=False)



area             filas=  119919  columnas=3


artist           filas=  567202  columnas=8
artist_type      filas=       6  columnas=2
country_area     filas=     258  columnas=1
event            filas=  117931  columnas=11
event_alias      filas=    2748  columnas=6
event_type       filas=       8  columnas=2
l_area_area      filas=  119851  columnas=3
l_artist_event   filas=  336057  columnas=3
l_event_place    filas=  104799  columnas=3
l_event_series   filas=   57162  columnas=3
place            filas=   80651  columnas=6
series           filas=   36068  columnas=4

--- place ---
+-----+------------------------------------+----------------------------------+----+-----+---------------------+
|id   |mbid                                |name                              |type|area |coordinates          |
+-----+------------------------------------+----------------------------------+----+-----+---------------------+
|11376|caa66bca-1a61-493c-90ef-342784f822c1|Dunedin Muso's Club               |2   |5430 |null                 |
|113

In [4]:
MARCADORES = ["", "\\N"]
def reporte_nulos(df, titulo):
    total = df.count()
    exprs = [F.sum((F.col(c).isNull() |
                    F.trim(F.col(c).cast("string")).isin(MARCADORES)).cast("int")).alias(c)
             for c in df.columns]
    fila = df.select(*exprs).collect()[0].asDict()
    print(f"\n{titulo}  (filas: {total})")
    for c in df.columns:
        k = int(fila[c]); print(f"   {c:16} {k:>8} ({round(100*k/total,1)}%)")

for name, df in mb.items():
    reporte_nulos(df, f"NULOS EN {name}")




NULOS EN area  (filas: 119919)
   id                      0 (0.0%)
   mbid                    0 (0.0%)
   name                    0 (0.0%)



NULOS EN artist  (filas: 567202)
   id                      0 (0.0%)
   mbid                    0 (0.0%)
   name                    0 (0.0%)
   sort_name               0 (0.0%)
   begin_year             56 (0.0%)
   end_year           426517 (75.2%)
   type                   39 (0.0%)
   area                68106 (12.0%)

NULOS EN artist_type  (filas: 6)
   id                      0 (0.0%)
   name                    0 (0.0%)

NULOS EN country_area  (filas: 258)
   area_id                 0 (0.0%)



NULOS EN event  (filas: 117931)
   id                      0 (0.0%)
   mbid                    0 (0.0%)
   name                    0 (0.0%)
   begin_year            590 (0.5%)
   begin_month          1561 (1.3%)
   begin_day            2129 (1.8%)
   end_year             1712 (1.5%)
   end_month            2706 (2.3%)
   end_day              3322 (2.8%)
   type                 5762 (4.9%)
   cancelled              36 (0.0%)

NULOS EN event_alias  (filas: 2748)
   id                      0 (0.0%)
   event_id                0 (0.0%)
   name                    0 (0.0%)
   locale               1385 (50.4%)
   type                 1110 (40.4%)
   sort_name               0 (0.0%)

NULOS EN event_type  (filas: 8)
   id                      0 (0.0%)
   name                    0 (0.0%)



NULOS EN l_area_area  (filas: 119851)
   id                      0 (0.0%)
   parent_area_id          0 (0.0%)
   child_area_id           0 (0.0%)



NULOS EN l_artist_event  (filas: 336057)
   id                      0 (0.0%)
   artist_id               0 (0.0%)
   event_id                0 (0.0%)

NULOS EN l_event_place  (filas: 104799)
   id                      0 (0.0%)
   event_id                0 (0.0%)
   place_id                0 (0.0%)

NULOS EN l_event_series  (filas: 57162)
   id                      0 (0.0%)
   event_id                0 (0.0%)
   series_id               0 (0.0%)



NULOS EN place  (filas: 80651)
   id                      0 (0.0%)
   mbid                    0 (0.0%)
   name                    0 (0.0%)
   type                 6708 (8.3%)
   area                 8344 (10.3%)
   coordinates         50503 (62.6%)

NULOS EN series  (filas: 36068)
   id                      0 (0.0%)
   mbid                    0 (0.0%)
   name                    0 (0.0%)
   type                    4 (0.0%)


In [5]:
pk = {  # clave primaria de cada tabla
 "area":"id","artist":"id","artist_type":"id","country_area":"area_id","event":"id",
 "event_alias":"id","event_type":"id","l_area_area":"id","l_artist_event":"id",
 "l_event_place":"id","l_event_series":"id","place":"id","series":"id",
}
for name, df in mb.items():
    total = df.count()
    dup = total - df.select(pk[name]).distinct().count()
    print(f"{name:16} PK={pk[name]:14} duplicados={dup:>5}  filas_exactas_dup={total - df.dropDuplicates().count()}")

# Claves de negocio en las tablas de relación (muchos-a-muchos)
print("\nRelaciones duplicadas por par:")
print(" l_artist_event (artist_id,event_id):",
      mb["l_artist_event"].count() - mb["l_artist_event"].dropDuplicates(["artist_id","event_id"]).count())
print(" l_event_place (event_id,place_id):",
      mb["l_event_place"].count() - mb["l_event_place"].dropDuplicates(["event_id","place_id"]).count())
print(" l_event_series (event_id,series_id):",
      mb["l_event_series"].count() - mb["l_event_series"].dropDuplicates(["event_id","series_id"]).count())



area             PK=id             duplicados=    0  filas_exactas_dup=0


artist           PK=id             duplicados=    0  filas_exactas_dup=0
artist_type      PK=id             duplicados=    0  filas_exactas_dup=0
country_area     PK=area_id        duplicados=    0  filas_exactas_dup=0


event            PK=id             duplicados=    0  filas_exactas_dup=0
event_alias      PK=id             duplicados=    0  filas_exactas_dup=0
event_type       PK=id             duplicados=    0  filas_exactas_dup=0


l_area_area      PK=id             duplicados=    0  filas_exactas_dup=0


l_artist_event   PK=id             duplicados=    0  filas_exactas_dup=0
l_event_place    PK=id             duplicados=    0  filas_exactas_dup=0
l_event_series   PK=id             duplicados=    0  filas_exactas_dup=0


[Stage 296:============================>                            (1 + 1) / 2]

place            PK=id             duplicados=    0  filas_exactas_dup=0


series           PK=id             duplicados=    0  filas_exactas_dup=0

Relaciones duplicadas por par:


 l_artist_event (artist_id,event_id): 5888
 l_event_place (event_id,place_id): 0
 l_event_series (event_id,series_id): 25


In [6]:
place = mb["place"]
total = place.count()

print("Distribución de 'type' (tipo de lugar):")
place.groupBy("type").count().orderBy(F.desc("count")).show(10)

print("Places con área conocida:",
      place.filter(F.col("area").isNotNull()).count(), f"de {total}")

# coordinates viene como '(lat,lon)' -> parseamos a numérico
place_coord = (place
    .withColumn("latitude",  F.regexp_extract("coordinates", r"\(([-0-9.]+),([-0-9.]+)\)", 1).cast("double"))
    .withColumn("longitude", F.regexp_extract("coordinates", r"\(([-0-9.]+),([-0-9.]+)\)", 2).cast("double")))
print("Places con coordenadas parseadas:",
      place_coord.filter(F.col("latitude").isNotNull()).count(), f"de {total}")
print("Coordenadas fuera de rango:",
      place_coord.filter(F.col("latitude").isNotNull() &
          ~(F.col("latitude").between(-90,90) & F.col("longitude").between(-180,180))).count())



Distribución de 'type' (tipo de lugar):


+----+-----+
|type|count|
+----+-----+
|   1|35601|
|   2|19585|
|null| 6708|
|   3| 4757|
|   6| 4136|
|   7| 2860|
|  44| 1495|
|  42| 1366|
|   5| 1318|
|   4|  888|
+----+-----+
only showing top 10 rows

Places con área conocida: 72307 de 80651
Places con coordenadas parseadas: 30146 de 80651


[Stage 356:============================>                            (1 + 1) / 2]

Coordenadas fuera de rango: 15


In [7]:
ev = mb["event"]; lep = mb["l_event_place"]; place = mb["place"]

print("Eventos totales:", ev.count())
print("Eventos con al menos un lugar:",
      lep.select("event_id").distinct().count())
print("Eventos con más de un lugar:",
      lep.groupBy("event_id").count().filter(F.col("count") > 1).count())

# Integridad referencial
print("l_event_place con event_id inexistente:",
      lep.join(ev.select(F.col("id").alias("event_id")), "event_id", "left_anti").count())
print("l_event_place con place_id inexistente:",
      lep.join(place.select(F.col("id").alias("place_id")), "place_id", "left_anti").count())



Eventos totales: 117931
Eventos con al menos un lugar: 103833
Eventos con más de un lugar: 530
l_event_place con event_id inexistente: 0
l_event_place con place_id inexistente: 0


In [8]:
# country_area marca las áreas que SON país (techo de la jerarquía).
# Propagamos ese país hacia abajo (ciudad/región) usando l_area_area, iterando por niveles.
laa = mb["l_area_area"].select("parent_area_id", "child_area_id")

# nivel 0: cada país se mapea a sí mismo
area_country = mb["country_area"].select(
    F.col("area_id"), F.col("area_id").alias("country_id")
)

for i in range(10):
    nuevos = (laa
        .join(area_country.withColumnRenamed("area_id", "parent_area_id"), "parent_area_id")
        .select(F.col("child_area_id").alias("area_id"), "country_id"))
    antes = area_country.count()
    area_country = area_country.unionByName(nuevos).dropDuplicates(["area_id"])
    despues = area_country.count()
    print(f"Iteración {i}: áreas con país = {despues}")
    if despues == antes:
        break

# nombre del país
area_country = area_country.join(
    mb["area"].select(F.col("id").alias("country_id"), F.col("name").alias("country_name")),
    "country_id", "left")
print("Áreas resolubles a país:", area_country.count())


[Stage 386:============================>                            (1 + 1) / 2]

Iteración 0: áreas con país = 4081


Iteración 1: áreas con país = 37428


[Stage 435:============================>                            (1 + 1) / 2]

Iteración 2: áreas con país = 117932


...

[Stage 864:>                                                        (0 + 2) / 2]

Iteración 6: áreas con país = 119893


Iteración 7: áreas con país = 119893


[Stage 1162:============================>                           (1 + 1) / 2]

Áreas resolubles a país: 119893


In [9]:
ev = mb["event"]; lep = mb["l_event_place"]; place = mb["place"]

# evento -> un lugar -> área del lugar -> país
ev_place = (lep.dropDuplicates(["event_id"])
    .join(place.select(F.col("id").alias("place_id"), "area"), "place_id", "left")
    .join(area_country.select("area_id", "country_name"),
          F.col("area") == F.col("area_id"), "left"))

n = ev.count()
con_pais = ev_place.filter(F.col("country_name").isNotNull()).count()
print("Eventos totales:", n)
print("Eventos resueltos a PAÍS:", con_pais, f"({round(100*con_pais/n,1)}%)")

print("\nTop países por cantidad de eventos:")
ev_place.groupBy("country_name").count().orderBy(F.desc("count")).show(12, truncate=False)

print("Eventos en Uruguay:",
      ev_place.filter(F.col("country_name") == "Uruguay").count())


Eventos totales: 117931
Eventos resueltos a PAÍS: 102079 (86.6%)

Top países por cantidad de eventos:


+--------------+-----+
|country_name  |count|
+--------------+-----+
|United States |38586|
|United Kingdom|11470|
|Germany       |10470|
|Japan         |6405 |
|Canada        |5215 |
|Netherlands   |4172 |
|France        |3893 |
|Belgium       |2549 |
|null          |1754 |
|Israel        |1518 |
|Spain         |1455 |
|Australia     |1361 |
+--------------+-----+
only showing top 12 rows



[Stage 1360:>                                                       (0 + 2) / 2]

Eventos en Uruguay: 11


In [10]:
series = mb["series"]; les = mb["l_event_series"]

print("Series totales:", series.count())
print("Distribución de 'type' de serie:")
series.groupBy("type").count().orderBy(F.desc("count")).show(10)

print("Eventos vinculados a una serie:", les.select("event_id").distinct().count())
print("Series distintas referenciadas:", les.select("series_id").distinct().count())

# Integridad
print("l_event_series con series_id inexistente:",
      les.join(series.select(F.col("id").alias("series_id")), "series_id", "left_anti").count())
print("l_event_series con event_id inexistente:",
      les.join(mb["event"].select(F.col("id").alias("event_id")), "event_id", "left_anti").count())


Series totales: 36068
Distribución de 'type' de serie:
+----+-----+
|type|count|
+----+-----+
|   1|18187|
|   2| 5374|
|   7| 2703|
|   8| 1868|
|  14| 1564|
|  80| 1330|
|   6| 1076|
|   3| 1070|
|   4|  798|
|   5|  624|
+----+-----+
only showing top 10 rows

Eventos vinculados a una serie: 52167
Series distintas referenciadas: 5519
l_event_series con series_id inexistente: 0
l_event_series con event_id inexistente: 0
